# Friends Demo — Phase 3: The Read Path

query embedding → similarity search → ranking → injection

We can't see raw embedding vectors (MemoryClient is hosted), but we can see and control the
real levers: `top_k`, `threshold`, `rerank`, and each result's `score`.

Same Maya/Jordan/Sam data, continued from Phase 2.


### 📋 Phase 3 Overview & Execution Flow

Phase 3 demonstrates how the **Read Path** ranks and filters memories in order:

- **Section 0 — Reconnect & Disable Decay**: Connects to `MemoryClient` and sets `decay=False` to isolate relevance scoring from recency.
- **Section 1 — Inspect Relevance Scores**: Queries *"What hobby is Maya doing these days?"* and prints composite similarity scores next to each retrieved memory.
- **Sections 2–3 — Threshold & Top-K Filtering**: Demonstrates how `threshold` score cutoffs and `top_k` limits eliminate prompt noise.
- **Section 4 — Contradiction Reranking**: Tests whether new facts (*"painting"*) outrank stale facts (*"pottery"*) during search.
- **Section 5 — Impact on Generation**: Demonstrates how different retrieval thresholds directly change the final LLM response.


## 0. Reconnect, and turn decay off

Decay reranks by recency at search time. We turn it off so this notebook is a clean test of
relevance ranking alone -- we'll turn it back on deliberately in Phase 4.


In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))
client.project.update(decay=False)

existing = client.get_all(filters={"user_id": "maya"})
print(f"{len(existing.get('results', []))} memories on file for Maya")

10 memories on file for Maya


## 1. Scores, not just text

In [2]:
query = "what hobby is Maya doing these days?"

results = client.search(query=query, filters={"user_id": "maya"})
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

0.270  User is looking for birthday gift ideas for Maya
0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.260  Maya went hiking on the weekend of July 31 to August 1, 2026
0.255  Maya asked the assistant for a birthday gift suggestion
0.224  Maya asked the user for a birthday gift suggestion
0.203  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.177  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays
0.162  User recently started attending a pottery class that takes place on Tuesdays
0.136  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing
0.218  Maya's pottery class moved from Tuesdays to Thursdays starting next month.


## 2. Threshold experiment

In [3]:
for threshold in [0.0, 0.3, 0.6, 0.9]:
    results = client.search(query=query, filters={"user_id": "maya"}, threshold=threshold)
    print(f"threshold={threshold} -> {len(results.get('results', []))} results")

threshold=0.0 -> 10 results
threshold=0.3 -> 10 results
threshold=0.6 -> 9 results
threshold=0.9 -> 9 results


In [4]:
results = client.search(query=query, filters={"user_id": "maya"}, threshold=0.5)
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

0.270  User is looking for birthday gift ideas for Maya
0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.260  Maya went hiking on the weekend of July 31 to August 1, 2026
0.255  Maya asked the assistant for a birthday gift suggestion
0.224  Maya asked the user for a birthday gift suggestion
0.203  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.177  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays
0.162  User recently started attending a pottery class that takes place on Tuesdays
0.136  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing
0.218  Maya's pottery class moved from Tuesdays to Thursdays starting next month.


## 3. top_k experiment

In [5]:
for k in [1, 3, 10]:
    results = client.search(query=query, filters={"user_id": "maya"}, top_k=k)
    print(f"top_k={k}:")
    for r in results.get("results", []):
        print(f"   {r['score']:.3f}  {r['memory']}")
    print()

top_k=1:
   0.270  User is looking for birthday gift ideas for Maya

top_k=3:
   0.270  User is looking for birthday gift ideas for Maya
   0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
   0.260  Maya went hiking on the weekend of July 31 to August 1, 2026

top_k=10:
   0.270  User is looking for birthday gift ideas for Maya
   0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
   0.260  Maya went hiking on the weekend of July 31 to August 1, 2026
   0.255  Maya asked the assistant for a birthday gift suggestion
   0.224  Maya asked the user for a birthday gift suggestion
   0.203  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
   0.177  User quit pottery a few weeks ago (around July 15, 2026) and switched to paint

## 4. Ranking, using the pottery/painting contradiction

Phase 2 found that adding "I switched to painting" doesn't necessarily delete the old
pottery fact. This is the read-path test of whether ranking compensates for that.


In [6]:
results = client.search(
    query="what hobby is Maya doing these days?",
    filters={"user_id": "maya"},
    top_k=10,
)

for r in results.get("results", []):
    marker = ""
    if "painting" in r["memory"].lower():
        marker = "  <-- painting (new)"
    elif "pottery" in r["memory"].lower():
        marker = "  <-- pottery (stale)"
    print(f"{r['score']:.3f}  {r['memory']}{marker}")

0.270  User is looking for birthday gift ideas for Maya
0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.260  Maya went hiking on the weekend of July 31 to August 1, 2026
0.255  Maya asked the assistant for a birthday gift suggestion
0.224  Maya asked the user for a birthday gift suggestion
0.203  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.177  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays  <-- painting (new)
0.162  User recently started attending a pottery class that takes place on Tuesdays  <-- pottery (stale)
0.136  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing
0.218  Maya's pottery class moved from Tuesdays to Thursdays starting next month.  <-- pottery (stale)


If painting consistently outranks pottery, that's evidence the conflict gets resolved at
**read time** via scoring, not at write time via deletion. If they're close together, or
pottery ranks higher, that's a real limitation worth flagging, not smoothing over.


## 5. Bonus: rerank

In [7]:
plain = client.search(query=query, filters={"user_id": "maya"}, top_k=5)
reranked = client.search(query=query, filters={"user_id": "maya"}, top_k=5, rerank=True)

print("rerank=False:")
for r in plain.get("results", []):
    print(f"   {r['score']:.3f}  {r['memory']}")

print("\nrerank=True:")
for r in reranked.get("results", []):
    print(f"   {r['score']:.3f}  {r['memory']}")

rerank=False:
   0.270  User is looking for birthday gift ideas for Maya
   0.267  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
   0.260  Maya went hiking on the weekend of July 31 to August 1, 2026
   0.255  Maya asked the assistant for a birthday gift suggestion
   0.224  Maya asked the user for a birthday gift suggestion

rerank=True:
   0.858  Maya's pottery class moved from Tuesdays to Thursdays starting next month.
   0.847  User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays
   0.772  Maya went hiking on the weekend of July 31 to August 1, 2026
   0.757  User recently started attending a pottery class that takes place on Tuesdays
   0.640  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift


## 6. Injection comparison: does what you retrieve change what gets said?

In [8]:
from lab_llm_config import complete

def answer_with_settings(question, user_id, **search_kwargs):
    results = client.search(query=question, filters={"user_id": user_id}, **search_kwargs)
    memories = [r["memory"] for r in results.get("results", [])]
    memory_block = "\n".join(f"- {m}" for m in memories)
    prompt = (
        "You're chatting with a friend. Use these facts if relevant, answer naturally "
        f"without mentioning 'stored memories':\n\n{memory_block}\n\nQuestion: {question}"
    )
    return complete(prompt), memories

<frozen abc>:106: DeprecationWarning: BaseAgentConfig is deprecated and will be removed in future versions. Config is now loaded via reflection so the separate config class is no longer needed.


In [9]:
question = "what hobby is Maya doing these days?"

settings = {
    "top_1": {"top_k": 1},
    "top_5": {"top_k": 5},
    "threshold_0.5": {"threshold": 0.5, "top_k": 10},
}

for label, kwargs in settings.items():
    answer, memories = answer_with_settings(question, user_id="maya", **kwargs)
    print(f"--- {label} ({len(memories)} memories used) ---")
    print(answer)
    print()

--- top_1 (1 memories used) ---
I’m not sure what Maya’s been into lately—maybe you could check in with her or think about what she’s mentioned enjoying recently. If you’re looking for a gift, a little hint about her current interests would help narrow it down!

--- top_5 (5 memories used) ---
It looks like Maya’s been getting into hiking lately.

--- threshold_0.5 (10 memories used) ---




## Wrap-up

1. Relevance is a number (`score`), and `threshold`/`top_k` are the two levers that turn it
   into an actual result set.
2. Section 4 is the direct test of whether ranking, not storage, resolves the pottery/painting
   contradiction.
3. Section 6 shows the part that's actually visible to a user: different retrieval settings
   produce different answers, not just different printed lists.

**Next up:** Phase 4 -- decay and reinforcement.
